## Libraries, Local Server, and Datasets

In [ ]:
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time
import os
import json

# optional
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_SAMPLES =  PROJECT_ROOT / "data" / "samples"

In [ ]:
client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio"  # dummy, required
)

In [ ]:
sp500 = pd.read_csv(DATA_PROCESSED / "sp500_link_table.csv", index_col=0)

In [ ]:
observations = pd.read_csv(DATA_RAW / "transcripts_final.csv", index_col=0)
test_sample = observations.sample(100, random_state=42)
del observations

## Prompt Setup

### Identifier

In [ ]:
prompt_identify = """
You are a financial analyst.

Your task is to evaluate the ENTIRE earnings call Q&A text.

-----------------------
OBJECTIVE
-----------------------

Identify the extent of forward-looking information (FLI) in the FULL text.

Forward-looking information refers to statements about:
- expectations, guidance, or forecasts
- future performance, plans, or strategy
- anticipated risks or opportunities
- outlook on markets, demand, or operations

Do NOT extract or quote specific parts of the text.
Do NOT segment the text.

Evaluate the Q&A as a continuous dialogue.

-----------------------
OUTPUT FORMAT
-----------------------

Return ONLY valid JSON:

{
  "forward_looking_intensity": 0.00
}

-----------------------
SCORING
-----------------------

0.00 = no forward-looking content  
0.25 = very limited forward-looking references  
0.50 = moderate forward-looking discussion  
0.75 = substantial forward-looking content  
1.00 = predominantly forward-looking discussion  

Feel free to score anywhere between these values based on your assessment of the overall forward-looking nature of the Q&A text. Do not round to the nearest quarter if the content does not fit those exact categories. Use your judgment to assign a score that best reflects the forward-looking intensity of the entire Q&A dialogue.

"""

### Evaluation Prompt

In [ ]:
prompt_evaluate = """
You are a financial analyst.

You are given a forward-looking intensity (FLI) score.

Your task is to evaluate the forward-looking economic content in the ENTIRE earnings call Q&A text.

-----------------------
OBJECTIVE
-----------------------

This is a CONDITIONAL evaluation:

1) The Q&A may contain forward-looking information.
2) Evaluate ONLY the forward-looking content in the text.
3) Ignore discussions that are purely historical or not related to future expectations.

Treat the Q&A as a continuous dialogue. Do NOT extract or segment text.

-----------------------
IMPORTANT RULE
-----------------------

Use the provided FLI score as guidance:

- If FLI = 0.00:
  → all numerical values must be 0.00  
  → all categorical fields must be "none"  

- If FLI > 0.00:
  → evaluate the forward-looking content proportionally  
  → low FLI should result in low (but not necessarily zero) scores  
  → high FLI should result in more informative and developed scores  

Do NOT force non-zero values. Scores should reflect the actual strength and usefulness of the forward-looking content.

-----------------------
DIMENSIONS
-----------------------

Evaluate the forward-looking content on:

- specificity: level of detail and precision  
- economic_substance: usefulness for decision-making  
- tone: sentiment (-1 to 1)  
- certainty: confidence and strength of statements  


-----------------------
CATEGORICAL RULES
-----------------------

Select EXACTLY ONE per field. DO NOT create new categories.

main_focus:
strategy OR demand OR costs OR revenue OR supply OR investment OR risk OR product OR market OR competition OR regulation OR operations

secondary_focus:
same list, must differ from main_focus OR "none"

IMPORTANT:
- Assign based ONLY on forward-looking discussion

managerial_horizon:
short_term OR medium_term OR long_term OR mixed OR none

overall_outlook:
negative OR neutral OR positive OR none

DO NOT assign categories based on historical discussion or non-forward-looking content. Focus solely on the forward-looking elements of the Q&A when determining these categorical fields.
-----------------------
OUTPUT FORMAT
-----------------------

Return ONLY valid JSON and DO NOT give explanations. Restrict yourself to the following output format: 

{
  "specificity": 0.00,
  "economic_substance": 0.00,
  "tone": 0.00,
  "certainty": 0.00,
  "context_summary": {
    "main_focus": "",
    "secondary_focus": "",
    "managerial_horizon": "",
    "overall_outlook": ""
  }
}
"""

## LLM Functions

In [ ]:
def build_messages(dateutc, text, prompt):
    
    context_block = f"""Context:
Date of event: {dateutc}

Q&A Text:
{text}
"""

    return [
        {"role": "system", "content": prompt},
        {"role": "user", "content": context_block}
    ]

In [ ]:
def run_sample(text, dateutc, model, prompt, max_tokens, temperature=0):
    start = time.time()

    try:
        response = client.chat.completions.create(
            model=model,
            messages=build_messages(dateutc, text, prompt),
            temperature=temperature,
            max_tokens=max_tokens  # IMPORTANT: increase for testing
        )

        end = time.time()

        output = response.choices[0].message.content

        return {
            "output": output,
            "time_sec": end - start,
            "tokens": response.usage.total_tokens,
            "finish_reason": response.choices[0].finish_reason
        }

    except Exception as e:
        return {
            "output": None,
            "time_sec": None,
            "tokens": None,
            "finish_reason": None,
            "error": str(e)
        }

In [ ]:
def run_two_step_sample(text, dateutc, model, prompt_identify, prompt_evaluate, max_tokens, temperature=0):

    # --- STEP 1: IDENTIFY ---
    res1 = run_sample(
        text=text,
        dateutc=dateutc,
        model=model,
        prompt=prompt_identify,
        max_tokens=max_tokens,
        temperature=temperature
    )

    fli_score = None

    # Try extracting FLI safely
    try:
        parsed = json.loads(res1["output"])
        fli_score = parsed.get("forward_looking_intensity", None)
    except:
        pass

    # --- STEP 2: EVALUATE ---
    # OPTIONAL: anchor step 2 with step 1 output
    if fli_score is not None:
        prompt_eval_final = f"""
Forward-looking intensity (from previous step): {fli_score}

{prompt_evaluate}
"""
    else:
        prompt_eval_final = prompt_evaluate

    res2 = run_sample(
        text=text,
        dateutc=dateutc,
        model=model,
        prompt=prompt_eval_final,
        max_tokens=max_tokens,
        temperature=temperature
    )

    return {
        "step1_output": res1["output"],
        "step1_time": res1["time_sec"],
        "step1_tokens": res1["tokens"],
        "step1_finish": res1["finish_reason"],

        "step2_output": res2["output"],
        "step2_time": res2["time_sec"],
        "step2_tokens": res2["tokens"],
        "step2_finish": res2["finish_reason"],

        "fli_score": fli_score
    }

In [ ]:
os.makedirs("benchmarks/partial", exist_ok=True)
os.makedirs("benchmarks/final", exist_ok=True)

def benchmark(
    df,
    model,
    prompt_identify,
    prompt_evaluate,
    n = None,
    max_tokens=2000,
    xid=None,
    temperature=0
):
    # --- Sampling ---
    if n is None or n > len(df):
        samples = df
    else:
        samples = df.sample(n, random_state=2000)

    results = []

    # --- Run identifiers ---
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_model = model.replace("/", "_")

    filename = f"benchmarks/final/{safe_model}_2step_n{n}_maxtok{max_tokens}_{timestamp}.csv"
    partial_filename = f"benchmarks/partial/{safe_model}_2step_n{n}_maxtok{max_tokens}_{timestamp}_partial.csv"

    # --- Main loop ---
    for i, row in enumerate(tqdm(samples.itertuples(index=False)), 1):

        text = row.transcript_text
        dateutc = getattr(row, "mostimportantdateutc", None)

        res = run_two_step_sample(
            text=text,
            dateutc=dateutc,
            model=model,
            prompt_identify=prompt_identify,
            prompt_evaluate=prompt_evaluate,
            max_tokens=max_tokens,
            temperature=temperature
        )

        # --- Safety ---
        if res is None:
            res = {}

        # Ensure all expected keys exist
        for key in [
            "step1_output", "step1_time", "step1_tokens", "step1_finish",
            "step2_output", "step2_time", "step2_tokens", "step2_finish",
            "fli_score"
        ]:
            res.setdefault(key, None)

        # --- Metadata ---
        res["transcript_id"] = getattr(row, "transcriptid", None)
        res["date"] = dateutc

        if xid is not None:
            res[xid] = getattr(row, xid, None)

        # --- Combined metrics ---
        res["total_time"] = (res["step1_time"] or 0) + (res["step2_time"] or 0)
        res["total_tokens"] = (res["step1_tokens"] or 0) + (res["step2_tokens"] or 0)

        results.append(res)

        # --- Partial save ---
        if i % 10 == 0:
            pd.DataFrame(results).to_csv(partial_filename, index=False)

    results_df = pd.DataFrame(results)

    # =========================
    # 🔍 DIAGNOSTICS (FIXED)
    # =========================

    n_total = len(results_df)

    # --- Length issues ---
    n_length_1 = (results_df["step1_finish"] == "length").sum()
    n_length_2 = (results_df["step2_finish"] == "length").sum()

    pct_length_1 = (results_df["step1_finish"] == "length").mean()
    pct_length_2 = (results_df["step2_finish"] == "length").mean()

    # --- Missing outputs ---
    n_missing_1 = results_df["step1_output"].isna().sum()
    n_missing_2 = results_df["step2_output"].isna().sum()

    pct_missing_1 = results_df["step1_output"].isna().mean()
    pct_missing_2 = results_df["step2_output"].isna().mean()

    # --- Time stats ---
    time_quantiles = results_df["total_time"].quantile([0.25, 0.5, 0.75, 0.95, 0.99])

    # --- Final save ---
    results_df.to_csv(filename, index=False)

    # =========================
    # 📊 PRINT RESULTS
    # =========================

    print("\n=== Benchmark Results (2-Step) ===")
    print(f"Saved to: {filename}")
    print(f"Total samples: {n_total}")

    print("\n--- Truncation ---")
    print(f"Step 1 length: {n_length_1} ({pct_length_1:.2%})")
    print(f"Step 2 length: {n_length_2} ({pct_length_2:.2%})")

    print("\n--- Missing Output ---")
    print(f"Step 1 missing: {n_missing_1} ({pct_missing_1:.2%})")
    print(f"Step 2 missing: {n_missing_2} ({pct_missing_2:.2%})")

    print("\n--- Time ---")
    print(f"Avg total time: {results_df.total_time.mean():.2f}s")

    scale_factor = 11009
    print(f"Estimated full runtime: {(results_df.total_time.mean() * scale_factor) / 3600:.2f} hours")

    print("\n=== Time Quantiles ===")
    for q, val in time_quantiles.items():
        print(f"{int(q*100)}th percentile: {val:.2f}s")

    print(f"Max time: {results_df.total_time.max():.2f}s")

    print("\n=== Token Usage ===")
    print(f"Avg tokens (total): {results_df.total_tokens.mean():.0f}")
    print(f"Min tokens (total): {results_df.total_tokens.min():.0f}")
    print(f"Max tokens (total): {results_df.total_tokens.max():.0f}")

    print("\n=== Finish Reason Distribution ===")
    print("Step 1:")
    print(results_df["step1_finish"].value_counts(normalize=True))

    print("\nStep 2:")
    print(results_df["step2_finish"].value_counts(normalize=True))

    return results_df

## MODELS

In [ ]:
test_sample = pd.read_excel(PROJECT_ROOT/'qna_validation20_dated.xlsx')

In [ ]:
test_sample.columns = ['transcriptid', 'question_order', 'transcript_text',
       'mostimportantdateutc']

In [ ]:
obs = len(test_sample)

print("Running LLaMA 3.1 8B Instruct benchmark...")
benchmark(test_sample, model="meta-llama-3.1-8b-instruct",n= obs, prompt_evaluate=prompt_evaluate, prompt_identify=prompt_identify, temperature=0.5, xid='question_order')
print("Done, LLaMA results are available in the benchmarks/final folder.")
